# NB_07 — SOURCE_06 Extraction v1.0

Complete `SOURCE_06` (PySMuRF tuning / channel assignment), validate it, write the canonical YAML, and export a ZIP.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, zipfile, yaml

REPOSITORY_URL="https://github.com/thinkthoughts/sensors-becker.git"
SOURCE_ID="SOURCE_06"
SCAFFOLD_FILENAME="SOURCE_06_pysmurf_channel_assignment.scaffold.yaml"
SOURCE_FILENAME="SOURCE_06_pysmurf_channel_assignment.yaml"

def find_repo_root():
    s=Path.cwd().resolve()
    candidates=[s,*s.parents,Path("/content/sensors-becker"),Path("/home/dan/sensors-becker"),Path.home()/"sensors-becker"]
    for c in candidates:
        if c.is_dir() and (c/"engineering_navigator").is_dir():
            return c
    if Path("/content").exists():
        target=Path("/content/sensors-becker")
        if not target.exists():
            subprocess.run(["git","clone",REPOSITORY_URL,str(target)],check=True)
        if (target/"engineering_navigator").is_dir():
            return target
    raise FileNotFoundError("Could not locate sensors-becker")

ROOT=find_repo_root()
sys.path.insert(0,str(ROOT))
SOURCE_DIR=ROOT/"engineering_navigator"/"multiplexed_readout"/"source_records"
SCAFFOLD_PATH=SOURCE_DIR/SCAFFOLD_FILENAME
SOURCE_PATH=SOURCE_DIR/SOURCE_FILENAME
EXPORT_DIR=ROOT/"exports"/SOURCE_ID
EXPORT_ZIP=ROOT/"exports"/f"{SOURCE_ID}_export.zip"
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
EXPORT_ZIP.parent.mkdir(parents=True,exist_ok=True)

print("Repository:",ROOT)
print("Scaffold:",SCAFFOLD_PATH.relative_to(ROOT))


## Registry check


In [ ]:
from tools.source_extractors.registry import EXTRACTORS, extract_source
print(sorted(EXTRACTORS))
if SOURCE_ID not in EXTRACTORS:
    raise ValueError(f"{SOURCE_ID} is not registered")
print("Registry validation: PASS")


## Load scaffold and extract


In [ ]:
scaffold=yaml.safe_load(SCAFFOLD_PATH.read_text(encoding="utf-8"))
if scaffold.get("source_id") != SOURCE_ID:
    raise ValueError("Wrong source_id")
completed=extract_source(SOURCE_ID,scaffold)
print("Extraction status:",completed.get("extraction_status"))
print("GDT status:",completed.get("gdt_applicability",{}).get("status"))


## Validate completed record


In [ ]:
required=[
    "authors","materials","design_variables","reported_values",
    "engineering_relationships","engineering_constraints",
    "future_questions","unreported_variables"
]
errors=[]
for field in required:
    if not completed.get(field):
        errors.append(f"{field} empty or missing")
if completed.get("record_status")!="evidence_extracted":
    errors.append("record_status")
if not str(completed.get("extraction_status","")).startswith("complete"):
    errors.append("extraction_status")

expected="real_integer_assignment_present_but_direct_GDT_not_established"
actual=completed.get("gdt_applicability",{}).get("status")
if actual!=expected:
    errors.append(f"GDT status {actual!r} != {expected!r}")

if errors:
    raise ValueError("SOURCE_06 validation failed: "+", ".join(errors))
print("Completed source-record validation: PASS")


## Inspect GDT-relevant implementation evidence


In [ ]:
gdt=completed["gdt_applicability"]
for item in gdt.get("candidate_integer_structures",[]):
    print("-",item.get("structure"),"→",item.get("gdt_status"))
print("\nPhysical exclusion rule:")
print(gdt.get("physical_exclusion_rule"))
print("\nMissing hypotheses:")
for item in gdt.get("missing_direct_hypotheses",[]):
    print("-",item)


## Write canonical YAML


In [ ]:
SOURCE_PATH.write_text(
    yaml.safe_dump(completed,sort_keys=False,allow_unicode=True,width=110),
    encoding="utf-8",
)
roundtrip=yaml.safe_load(SOURCE_PATH.read_text(encoding="utf-8"))
if roundtrip != completed:
    raise ValueError("YAML round-trip changed SOURCE_06")
print("Wrote:",SOURCE_PATH.relative_to(ROOT))


## Build export


In [ ]:
shutil.rmtree(EXPORT_DIR,ignore_errors=True)
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
shutil.copy2(SOURCE_PATH,EXPORT_DIR/SOURCE_FILENAME)

manifest={
    "source_id":SOURCE_ID,
    "title":completed.get("title"),
    "extraction_status":completed.get("extraction_status"),
    "files":[SOURCE_FILENAME],
    "counts":{
        "authors":len(completed.get("authors",[])),
        "materials":len(completed.get("materials",[])),
        "design_variables":len(completed.get("design_variables",[])),
        "reported_values":len(completed.get("reported_values",[])),
        "engineering_relationships":len(completed.get("engineering_relationships",[])),
        "engineering_constraints":len(completed.get("engineering_constraints",[])),
        "gdt_candidate_integer_structures":len(completed.get("gdt_applicability",{}).get("candidate_integer_structures",[])),
    },
    "gdt_applicability_status":completed.get("gdt_applicability",{}).get("status"),
}
(EXPORT_DIR/"manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
if EXPORT_ZIP.exists(): EXPORT_ZIP.unlink()
with zipfile.ZipFile(EXPORT_ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(EXPORT_DIR.iterdir()):
        if p.is_file(): z.write(p,arcname=p.name)
print(json.dumps(manifest,indent=2))
print("Export:",EXPORT_ZIP)


## Download ZIP in Colab


In [ ]:
try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Colab.")


## Handoff

```text
SOURCE_06 scaffold
  ↓
source_06.py
  ↓
registry
  ↓
NB_07 extraction
  ↓
SOURCE_06 canonical YAML
  ↓
multiplexed_readout Engineering Object refresh
  ↓
combined SOURCE_05 + SOURCE_06 GDT applicability notebook
```
